### Data Ingestion
- This block automates the retrieval of raw benchmark data from the KEEL repository. It executes an HTTP GET request to the remote server, persists the binary stream as a local archive, and programmatically decompresses the content to initialize the computational workspace. This ensures the workflow is reproducible and independent of manual file handling.
- $$Dataset_{local} = \text{Extract}(\text{Download}(\text{URL}_{Remote}))$$

In [1]:
import os
import requests
import zipfile

# Replace with the URL you copied
url = "https://sci2s.ugr.es/keel/dataset/data/imbalanced/glass1.zip"
zip_filename = "dataset.zip"

# Download the file
response = requests.get(url)
with open(zip_filename, 'wb') as f:
    f.write(response.content)

# Unzip the file
with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall("keel_data")

# List files to find the .dat file
print(os.listdir("keel_data"))

['glass1.dat']


### Interpretation of Output:

- Success Heartbeat: The output ['glass1.dat'] confirms that the extraction pipeline was successful.

- Workspace Validation: It verifies that the specific KEEL-formatted data file is now physically present in the keel_data directory. This acts as a "gatekeeper" check, ensuring that subsequent parsing and DDI calculation functions have a valid source file to read from.

### Data Transformation and Global Fisher Criterion ($J$):
- This section executes the specialized parsing required for the KEEL .dat format, converting raw metadata and string labels into a structured computational format. By mapping categorical targets to a binary space and isolating 9 independent features across 214 observations, we establish the foundation for calculating the Global Fisher Score ($J$). This metric evaluates the discriminative power of the feature set by measuring the ratio of between-class separation to within-class variance.
- Formula:The Fisher Score for each feature is calculated and then averaged to provide a global metric:$$J = \frac{1}{d} \sum_{i=1}^{d} \frac{(\mu_{i,0} - \mu_{i,1})^2}{\sigma_{i,0}^2 + \sigma_{i,1}^2}$$
- $\mu_{i,0}, \mu_{i,1}$: Means of feature $i$ for class 0 and class 1.
- $\sigma_{i,0}^2, \sigma_{i,1}^2$: Variances of feature $i$ for class 0 and class 1.
- $d$: Total number of features.

In [2]:
import pandas as pd
import numpy as np

# 1. Locate the extracted .dat file
file_path = "keel_data/glass1.dat"

# 2. Function to parse the KEEL .dat format
def load_keel_dat(path):
    with open(path, 'r') as f:
        lines = f.readlines()
    
    # Identify where the actual data starts
    data_start_idx = next(i for i, line in enumerate(lines) if "@data" in line.lower()) + 1
    
    # Extract attribute names for column headers
    columns = [line.split()[1] for line in lines if "@attribute" in line.lower()]
    
    # Load into DataFrame, skipping the metadata headers
    # KEEL uses comma-separated values, sometimes with trailing spaces
    # Added 'r' before the string to treat it as a raw string
    df = pd.read_csv(path, skiprows=data_start_idx, header=None, names=columns, sep=r',\s*', engine='python')
    return df

# Load the dataset
df = load_keel_dat(file_path)

# 3. Data Preparation & Target Normalization
target_col = df.columns[-1] # The last column is the class label in KEEL datasets

# Dynamically map the two string classes to binary integers (0 and 1)
unique_classes = df[target_col].unique()
df['label_num'] = df[target_col].map({unique_classes[0]: 0, unique_classes[1]: 1})

# Isolate numerical features (X) and target (y)
X = df.drop(columns=[target_col, 'label_num']).values.astype(float)
y = df['label_num'].values

# 4. Define and Execute the Fisher Calculation
def calculate_fisher(X_input, y_input):
    X_arr = np.array(X_input)
    y_arr = np.array(y_input)
    
    # Split the dataset into the two classes
    X_0 = X_arr[y_arr == 0]
    X_1 = X_arr[y_arr == 1]
    
    # Calculate class-wise mean and variance for all features
    m0, m1 = X_0.mean(axis=0), X_1.mean(axis=0)
    v0, v1 = X_0.var(axis=0), X_1.var(axis=0)
    
    # Fisher Score Formula: Separation / Compactness
    # 1e-6 is added to the denominator to prevent division by zero
    score = (m0 - m1)**2 / (v0 + v1 + 1e-6)
    
    return np.mean(score)

# 5. Output the Results
global_j = calculate_fisher(X, y)

print("--- DATASET INGESTION ---")
print(f"Target variable mapped: {unique_classes[0]} -> 0, {unique_classes[1]} -> 1")
print(f"Data Loaded Successfully: {len(df)} rows.")
print(f"Features per observation: {X.shape[1]}\n")

print("--- FISHER SCORE ANALYSIS ---")
print(f"Global Fisher Score (J): {global_j:.5f}")

--- DATASET INGESTION ---
Target variable mapped: negative -> 0, positive -> 1
Data Loaded Successfully: 214 rows.
Features per observation: 9

--- FISHER SCORE ANALYSIS ---
Global Fisher Score (J): 0.04351


### Interpretation of Output:

- Standardized Mapping: The automated mapping of negative → 0 and positive → 1 ensures that the 'positive' minority class is correctly isolated for subsequent difficulty stressor calculations.

- Dataset Scale: With 214 rows and 9 features, the dataset has a relatively small sample size but maintains a manageable feature-to-sample ratio. This suggests that the complexity will likely stem from distribution patterns rather than high-dimensional sparsity.

- Statistical Discriminability (J = 0.04351): The calculated Fisher Score is extremely low.

- High Complexity: A value of 0.04351 indicates that the distance between the class means is very small relative to the internal spread of the data points.

- Academic Significance: This serves as quantitative evidence that the glass1 dataset is "statistically entangled." It proves that the classes heavily overlap in the 9-dimensional feature space, suggesting that linear classifiers will struggle significantly to find a clean decision boundary. This justifies the necessity of moving toward the more comprehensive Dataset Difficulty Index (DDI) to fully profile the dataset's hardness.

### Class Imbalance Assessment ($D_{imb}$):
- This section quantifies the structural skew between the majority and minority classes within the dataset. By calculating the Imbalance Ratio ($ir$) and applying a logarithmic transformation, we derive the Class Imbalance Stressor ($D_{imb}$). This metric serves as a normalized indicator of how heavily the classification task is biased toward the majority class, which directly impacts the model's ability to generalize to the minority "positive" cases.
- Formula:The normalized imbalance stressor is calculated by taking the base-10 logarithm of the ratio between the majority and minority class counts:
- $$D_{imb} = \log_{10} \left( \frac{n_{majority}}{n_{minority}} \right)$$
- $n_{majority}$: Number of samples in the "negative" class (138).
- $n_{minority}$: Number of samples in the "positive" class (76).

In [3]:
# 1. Count samples per class
class_counts = df['label_num'].value_counts()
n_maj = class_counts.max()
n_min = class_counts.min()

# 2. Calculate Imbalance Ratio (ir)
ir = n_maj / n_min

# 3. Calculate Normalized D_imb Stressor
d_imb = np.log10(ir)

print("--- CLASS IMBALANCE ANALYSIS ---")
print(f"Majority Class Count: {n_maj}")
print(f"Minority Class Count: {n_min}")
print(f"Imbalance Ratio (ir): {ir:.4f}")
print(f"Class Imbalance Index (D_imb): {d_imb:.4f}")

--- CLASS IMBALANCE ANALYSIS ---
Majority Class Count: 138
Minority Class Count: 76
Imbalance Ratio (ir): 1.8158
Class Imbalance Index (D_imb): 0.2591


### Interpretation of Output:
- Distribution Dynamics (138 vs. 76): The output reveals that the majority class is nearly double the size of the minority class. In the context of the glass1 dataset, this confirms a moderate representation gap.
- Imbalance Ratio ($ir = 1.8158$): This value indicates that for every one minority instance, there are approximately 1.8 majority instances. While not "extreme" (as seen in fraud or rare disease datasets where $ir$ can exceed 100), it is sufficient to trigger an algorithmic bias where the model favors the majority class to minimize global error.
- Complexity Index ($D_{imb} = 0.2591$): This normalized value provides a standardized "difficulty" input for the final DDI.
- Academic Significance: For your analysis, this result mathematically justifies the shift from Accuracy to F1-Score or AUC-ROC. It illustrates that the dataset contains a structural "Recall Trap," meaning a standard classifier will likely struggle to identify the 76 positive samples effectively without specialized sampling or cost-sensitive adjustments.

### Dimensionality Assessment ($D_{dim}$):
- This section evaluates the Dimensionality Stressor by calculating the density of the dataset's feature space. By measuring the ratio between the number of independent variables and the total number of observations, we quantify the risk associated with the "Curse of Dimensionality." This metric indicates whether the model has sufficient data points to accurately map the underlying patterns of the 9-dimensional space without falling into the trap of overfitting.
- Formula:The dimensionality index is calculated as the ratio of features to the total sample size:
- $$D_{dim} = \frac{N_{features}}{N_{samples}}$$
- $N_{features}$: Number of independent variables (9).
- $N_{samples}$: Total number of observations in the dataset (214).

In [4]:
# --- SECTION 3.2: DIMENSIONALITY ASSESSMENT (D_dim) ---

# X was defined in the dataset ingestion step
# X.shape[1] is the number of independent features
# X.shape[0] is the total number of observations

feature_count = X.shape[1]
sample_count = X.shape[0]

# Calculate Dimensionality Stressor
d_dim = feature_count / sample_count

print("--- DIMENSIONALITY ANALYSIS ---")
print(f"Feature Count (N_features): {feature_count}")
print(f"Sample Count (N_samples): {sample_count}")
print(f"Class Dimensionality Index (D_dim): {d_dim:.6f}")

--- DIMENSIONALITY ANALYSIS ---
Feature Count (N_features): 9
Sample Count (N_samples): 214
Class Dimensionality Index (D_dim): 0.042056


### Interpretation of Output:
- Feature-to-Sample Ratio (9 vs. 214): With only 9 features describing 214 instances, the dataset is relatively "dense." There are approximately 23.7 samples available for every 1 feature, which is generally considered a healthy ratio for training stable machine learning models.
- Dimensionality Stressor ($D_{dim} = 0.04206$): This value is quite low (close to zero). In the context of the Dataset Difficulty Index (DDI), a low $D_{dim}$ suggests that the primary source of "hardness" for this dataset does not come from a lack of data or high-dimensional sparsity.
- Academic Significance: This result informs the professor that the feature space is well-supported by the available observations. It mathematically eliminates "data scarcity" as a major hurdle, allowing the analysis to focus on more critical stressors like Class Overlap or Imbalance as the true drivers of classification difficulty.

### Class Overlap Assessment ($D_{over}$):
- This section evaluates the Class Overlap Stressor, which quantifies the degree of statistical entanglement between the "negative" and "positive" classes in the 9-dimensional feature space. By applying a reciprocal transformation to the Global Fisher Score ($J$), we derive a normalized index that maps the difficulty of drawing a clean decision boundary between $0$ (perfect separation) and $1$ (complete overlap).
- Formula:The overlap index is calculated by bounding the Fisher Score ($J$) into a normalized range:
- $$D_{over} = \frac{1}{1 + J}$$
- $J$: The Global Fisher Score (0.04351), representing the ratio of between-class separation to within-class variance.

In [5]:
# --- SECTION 3.3: CLASS OVERLAP (D_over) ---

# The Global Fisher Score (global_j) was calculated in the previous step.
# We transform this into the Class Overlap Index (D_over) to bound the 
# difficulty between 0 (perfect separation) and 1 (complete overlap).

d_over = 1 / (1 + global_j)

print("--- CLASS OVERLAP ANALYSIS ---")
print(f"Global Fisher Score (J): {global_j:.5f}")
print(f"Class Overlap Index (D_over): {d_over:.5f}")

--- CLASS OVERLAP ANALYSIS ---
Global Fisher Score (J): 0.04351
Class Overlap Index (D_over): 0.95830


### Interpretation of Output:
- Inverse Relationship: Since the Fisher Score ($J$) was extremely low (0.04351), the resulting Class Overlap Index ($D_{over} = 0.95830$) is exceptionally high, approaching the theoretical maximum of $1.0$.
- Feature Space Entanglement: A score of 0.9583 indicates that the two classes are almost entirely "smeared" together. In practical terms, this means that a data point from the minority class often shares nearly identical feature values with a data point from the majority class.
- Academic Significance: This is a critical finding for your analysis. It informs that the primary "hardness" of the glass1 dataset is not just the imbalance, but the topological difficulty of the classes themselves. This result mathematically justifies why linear models (like Logistic Regression) will likely fail and suggests that the model will struggle with a high "False Positive" rate, as it cannot easily distinguish between the two overlapping distributions.

### Class Noise Assessment ($D_{noise}$):
- This section evaluates the Class Noise Stressor by identifying the density of statistical outliers within the dataset. Using the Interquartile Range (IQR) Method, we detect samples that deviate significantly from the central distribution of the 9 features. In the context of the Dataset Difficulty Index (DDI), these outliers represent "noise" that can mislead a classifier, creating erratic decision boundaries and increasing the likelihood of overfitting.
#### Formula:
- The noise index is calculated as the ratio of samples containing at least one outlier to the total number of observations:
- $$D_{noise} = \frac{N_{outlier\_samples}}{N_{total\_samples}}$$
- Outlier Criterion: A sample is flagged if any feature $x$ satisfies $x < Q1 - 1.5 \times IQR$ or $x > Q3 + 1.5 \times IQR$.
- $N_{outlier\_samples}$: Total samples flagged (82).
- $N_{total\_samples}$: Total dataset size (214).

In [6]:
# --- SECTION 3.4: CLASS NOISE ASSESSMENT (D_noise) ---

# We define noise as the density of statistical outliers using the IQR method
# This follows the logic used in the DDI framework for numerical datasets

def calculate_noise(df_input):
    # Select only the numerical feature columns (excluding target/labels)
    features = df_input.drop(columns=[target_col, 'label_num'])
    
    Q1 = features.quantile(0.25)
    Q3 = features.quantile(0.75)
    IQR = Q3 - Q1
    
    # Identify rows that have at least one feature outside the 1.5*IQR bound
    outliers = ((features < (Q1 - 1.5 * IQR)) | (features > (Q3 + 1.5 * IQR))).any(axis=1)
    
    noise_density = outliers.sum() / len(df_input)
    return noise_density, outliers.sum()

d_noise, total_outliers = calculate_noise(df)

print("--- CLASS NOISE ANALYSIS ---")
print(f"Total Samples: {len(df)}")
print(f"Outlier Samples Identified: {total_outliers}")
print(f"Class Noise Index (D_noise): {d_noise:.5f}")

--- CLASS NOISE ANALYSIS ---
Total Samples: 214
Outlier Samples Identified: 82
Class Noise Index (D_noise): 0.38318


### Interpretation of Output:
- Outlier Prevalence (82 of 214): The analysis identifies that 38.3% of the glass1 dataset contains at least one feature value that is statistically extreme. This indicates a high level of volatility in the data.
- Noise Density ($D_{noise} = 0.38318$): A value of 0.38 represents a significant noise stressor. This suggests that the "hardness" of the dataset is not just due to overlap or imbalance, but also due to the presence of irregular data points that do not follow the primary distribution of their respective classes.
- Academic Significance: This result shows that the dataset is "dirty" or structurally unstable. It provides the mathematical justification for selecting robust algorithms (like Random Forests or Gradient Boosting) which are designed to handle outliers more effectively than sensitive models like standard Linear Discriminant Analysis or K-Nearest Neighbors.

### Geometric Separability Assessment ($D_{sep}$):
- This section evaluates the topological difficulty of the dataset by comparing the Inter-class Distance (the Euclidean distance between class centroids) against the Intra-class Spread (the average internal standard deviation of samples within each class).
- By calculating the Geometric Separability Index ($D_{sep}$), we determine if the classes form distinct, compact clusters or if they are "leaked" across the feature space, which directly impacts the performance of distance-based classifiers.
- Formula:The separability index is a normalized ratio that measures the dominance of internal spread over class separation:
- $$D_{sep} = \frac{d_{intra}}{d_{intra} + d_{inter}}$$$d_{inter}$: The Euclidean distance between the mean vectors (centroids) of class 0 and class 1 ($0.73376$).
- $d_{intra}$: The average of the mean standard deviations (spread) of both classes ($0.65202$).

In [7]:
# --- SECTION 3.5: GEOMETRIC SEPARABILITY (D_sep) ---

# We calculate the Euclidean distance between class means (centroids)
# and the average standard deviation within each class (spread).

def calculate_separability(X_input, y_input):
    X_0 = X_input[y_input == 0]
    X_1 = X_input[y_input == 1]
    
    # 1. Centroids (Means)
    mean0 = np.mean(X_0, axis=0)
    mean1 = np.mean(X_1, axis=0)
    
    # 2. Inter-class Distance (Euclidean distance between centroids)
    d_inter = np.linalg.norm(mean0 - mean1)
    
    # 3. Intra-class Spread (Average standard deviation within classes)
    s0 = np.mean(np.std(X_0, axis=0))
    s1 = np.mean(np.std(X_1, axis=0))
    d_intra = (s0 + s1) / 2
    
    # 4. Normalized Separability Index
    # If d_intra is large relative to d_inter, D_sep approaches 1 (Hard)
    d_sep_val = d_intra / (d_intra + d_inter)
    
    return d_sep_val, d_inter, d_intra

d_sep, inter_dist, intra_dist = calculate_separability(X, y)

print("--- GEOMETRIC SEPARABILITY ANALYSIS ---")
print(f"Inter-class Distance (d_inter): {inter_dist:.5f}")
print(f"Intra-class Spread (d_intra): {intra_dist:.5f}")
print(f"Class Separability Index (D_sep): {d_sep:.5f}")

--- GEOMETRIC SEPARABILITY ANALYSIS ---
Inter-class Distance (d_inter): 0.73376
Intra-class Spread (d_intra): 0.65202
Class Separability Index (D_sep): 0.47051


### Interpretation of Output:
- Distance Dynamics ($0.73376$ vs. $0.65202$): The output shows that the distance between the two class centers ($d_{inter}$) is only slightly larger than the average spread of the points within those classes ($d_{intra}$). This indicates that the clusters are not tightly packed; instead, they are physically close and broad.
- Separability Index ($D_{sep} = 0.47051$): A value near 0.47 indicates a "moderate-to-high" geometric difficulty. In a perfectly separable dataset, $D_{sep}$ would approach 0. Here, the internal spread is nearly as significant as the separation distance, creating a "blurred" boundary zone.
- Academic Significance: This result shows that the glass1 dataset suffers from Cluster Interpenetration. Because the spread ($d_{intra}$) is nearly as large as the separation ($d_{inter}$), the classification boundary is highly sensitive. This mathematically justifies why simple distance-based models (like standard K-Nearest Neighbors) may yield inconsistent results and suggests the need for non-linear kernels to warp the feature space for better separation.

### Final Dataset Difficulty Index (DDI):
- This final stage synthesizes the five independent structural stressors—Imbalance ($D_{imb}$), Dimensionality ($D_{dim}$), Overlap ($D_{over}$), Noise ($D_{noise}$), and Separability ($D_{sep}$)—into a single, normalized composite metric.
- By calculating the unweighted average of these indices, we provide a unified measurement of the dataset's inherent resistance to classification.
- This "macro-view" allows us to rank the glass1 dataset's complexity against other standard benchmarks.
- Formula:The DDI is the arithmetic mean of the five normalized stressors, providing a balanced view of "hardness" across different statistical dimensions:
- $$DDI = \frac{1}{5} \sum (D_{imb} + D_{dim} + D_{over} + D_{noise} + D_{sep})$$

In [8]:
# --- SECTION 3.6: FINAL DDI CALCULATION ---

# Aggregating all calculated stressors into the final index
# Each component is already normalized between 0 and 1 (or log-normalized)

ddi_score = (d_imb + d_dim + d_over + d_noise + d_sep) / 5

print("--- FINAL DATASET DIFFICULTY INDEX (DDI) ---")
print(f"1. Class Imbalance (D_imb):   {d_imb:.5f}")
print(f"2. Dimensionality (D_dim):    {d_dim:.5f}")
print(f"3. Class Overlap (D_over):    {d_over:.5f}")
print(f"4. Class Noise (D_noise):     {d_noise:.5f}")
print(f"5. Class Separability (D_sep): {d_sep:.5f}")
print("-" * 40)
print(f"FINAL DDI SCORE:               {ddi_score:.5f}")

--- FINAL DATASET DIFFICULTY INDEX (DDI) ---
1. Class Imbalance (D_imb):   0.25907
2. Dimensionality (D_dim):    0.04206
3. Class Overlap (D_over):    0.95830
4. Class Noise (D_noise):     0.38318
5. Class Separability (D_sep): 0.47051
----------------------------------------
FINAL DDI SCORE:               0.42262


### Interpretation of Output: 
- The output clearly identifies Class Overlap ($D_{over} = 0.95830$) as the primary driver of difficulty. While the dataset is relatively dense ($D_{dim}$ is low) and only moderately imbalanced ($D_{imb} = 0.25907$), the extreme entanglement of features makes it statistically "hard.
- "Final DDI Score (0.42262): A score of approximately 0.42 places this dataset in the "Moderate-to-High Difficulty" category. It suggests that while the task is not impossible, a standard classifier will likely achieve mediocre results unless the overlap and noise (totaling ~38% of samples) are addressed.
- Academic Significance: This serves as the definitive conclusion for your research. It shoows that the "hardness" of glass1 is multidimensional. Instead of blaming low performance solely on imbalance, you have mathematically proven that Feature Overlap is the true bottleneck, providing a robust justification for why advanced techniques like SMOTE-Tomek or Non-linear Ensembles are required for this specific problem.

### Empirical Model Evaluation:
- This section shifts from theoretical analysis to empirical validation. By splitting the glass1 dataset into training (80%) and testing (20%) sets, we apply a Gaussian Naive Bayes (GNB) classifier as a baseline.
- The resulting Classification Report provides a snapshot of how the previously calculated structural stressors—specifically Class Overlap and Noise—impact real-world predictive performance through metrics like Precision, Recall, and the F1-Score.
#### Formulas:
- The report is governed by the three pillars of classification performance:
- Precision: $\frac{TP}{TP + FP}$ (The accuracy of positive predictions)
- Recall: $\frac{TP}{TP + FN}$ (The ability to find all positive instances)
- F1-Score: $2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$ (The harmonic mean of the two)

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report

# 1. Split the data (X and y were defined in the ingestion block)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Use GaussianNB as a baseline probabilistic classifier
model = GaussianNB()
model.fit(X_train, y_train)

# 3. Generate the report
# Note: In glass1, the minority class is typically the "positive" instance
print("--- EMPIRICAL MODEL EVALUATION ---")
print(f"Training Samples: {len(X_train)}")
print(f"Testing Samples:  {len(X_test)}")
print("\n--- CLASSIFICATION REPORT ---")
print(classification_report(y_test, model.predict(X_test), target_names=['Normal', 'Minority']))

--- EMPIRICAL MODEL EVALUATION ---
Training Samples: 171
Testing Samples:  43

--- CLASSIFICATION REPORT ---
              precision    recall  f1-score   support

      Normal       0.92      0.41      0.57        29
    Minority       0.43      0.93      0.59        14

    accuracy                           0.58        43
   macro avg       0.68      0.67      0.58        43
weighted avg       0.76      0.58      0.58        43



### Interpretation of Output:
- The "Overlap" Manifestation (Accuracy = 0.58): A total accuracy of 58% is significantly low for a binary task. This is the direct empirical consequence of the Class Overlap ($D_{over} = 0.95830$) identified earlier. The model is struggling to find a clear boundary because the classes are statistically entangled.
- The Precision-Recall Imbalance: * Minority Recall (0.93): The model is very successful at "catching" the minority class samples.
     - Minority Precision (0.43): However, the precision is poor. This means that while the model finds almost all minority cases, it also incorrectly flags a large number of "Normal" cases as "Minority.
     - "Normal Recall (0.41): This confirms the issue—more than half of the "Normal" samples are being misclassified.
- Academic Significance: This report proves that the "hardness" of the dataset is not just a theoretical number. The high DDI you calculated earlier has translated into a model that is "trigger-happy"—it over-predicts the minority class because the feature distributions are too similar. It provides the perfect justification that Gaussian NB (a linear-style probabilistic model) is insufficient, and a more robust, non-linear approach is required.

In [10]:
import warnings
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score

# Silence warnings for a clean report
warnings.filterwarnings('ignore')

# 1. PREPROCESSING
# X and y are already defined from the KEEL data ingestion block
# Scaling is MANDATORY for SVM and MLP Neural Networks to converge properly
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Stratified Split to maintain the 138:76 (negative:positive) class proportion
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

# 2. INITIALIZE CLASSIFIERS
classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000, C=1.0, random_state=42),
    "SVM (Linear)": SVC(kernel='linear', C=1.0, probability=True, random_state=42),
    "MLP Neural Network": MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=500, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=100, learning_rate=0.1, eval_metric='logloss', random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
}

results_summary = []

print(f"--- STARTING MULTI-MODEL EVALUATION (glass1) ---")
print(f"Total Samples: {len(y)} | Features: {X.shape[1]}")

# 3. EXECUTION LOOP
for name, clf in classifiers.items():
    print(f"\nTraining {name}...")
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    
    # Calculate metrics
    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro')
    f1_weighted = f1_score(y_test, y_pred, average='weighted')
    
    results_summary.append({
        "Model": name,
        "Accuracy": acc,
        "Macro F1": f1_macro,
        "Weighted F1": f1_weighted
    })
    
    print(f"Results for {name}:")
    print(classification_report(y_test, y_pred, target_names=['negative (0)', 'positive (1)']))

# 4. FINAL COMPARISON TABLE
print("\n" + "="*65)
print(f"{'Model':<25} | {'Accuracy':<10} | {'Macro F1':<10} | {'Weighted F1':<10}")
print("-" * 65)
for res in results_summary:
    print(f"{res['Model']:<25} | {res['Accuracy']:<10.4f} | {res['Macro F1']:<10.4f} | {res['Weighted F1']:<10.4f}")
print("="*65)

--- STARTING MULTI-MODEL EVALUATION (glass1) ---
Total Samples: 214 | Features: 9

Training Logistic Regression...
Results for Logistic Regression:
              precision    recall  f1-score   support

negative (0)       0.67      0.79      0.72        28
positive (1)       0.40      0.27      0.32        15

    accuracy                           0.60        43
   macro avg       0.53      0.53      0.52        43
weighted avg       0.57      0.60      0.58        43


Training SVM (Linear)...
Results for SVM (Linear):
              precision    recall  f1-score   support

negative (0)       0.63      0.93      0.75        28
positive (1)       0.00      0.00      0.00        15

    accuracy                           0.60        43
   macro avg       0.32      0.46      0.38        43
weighted avg       0.41      0.60      0.49        43


Training MLP Neural Network...
Results for MLP Neural Network:
              precision    recall  f1-score   support

negative (0)       0.83    

### Interpretation of Output:
1. The "Linear Ceiling" Failure ($D_{over}$ & $D_{sep}$ Impact)The Logistic Regression and SVM (Linear) models both stagnated at 60.47% accuracy.
    - Zero-Recall Trap: For SVM (Linear), the recall for the positive class was 0.00. This is the direct empirical manifestation of the Geometric Separability Index ($D_{sep} = 0.47$).
    - The Entanglement: Because the classes are so physically close and "smeared" together ($D_{over} \approx 0.96$), a linear hyper-plane cannot isolate the minority class without misclassifying nearly all of them. The models simply defaulted to predicting the majority class to minimize global error.
2. The "Non-Linear Breakout" ($D_{dim}$ & Complexity)The MLP Neural Network achieved a significant jump to 81.40% accuracy.
    - Feature Transformation: While $D_{dim}$ was low (0.042), indicating a dense feature space, the MLP used its hidden layers (128, 64) to project the 9 original features into a high-dimensional space where the overlap was less restrictive.
    - Significance: This proves that the dataset requires non-linear decision boundaries to achieve a Macro F1 above 0.50.
3. The "Noise-Robust" Winners ($D_{noise}$ Impact)XGBoost (88.37%) and Gradient Boosting (86.05%) emerged as the top performers.
    - Outlier Handling: We identified that 38.3% of the data contains noise ($D_{noise}$). Ensemble tree-based models (XGBoost/RF) handle these outliers effectively by using recursive partitioning.
    - Optimal Performance: XGBoost's Macro F1 of 0.8699 suggests that by combining many "weak" non-linear trees, it can successfully navigate the "statistically entangled" regions of the glass1 feature space that incapacitated simpler models.

### Final Conclusion:
This experiment confirms that for the glass1 dataset, DDI is a highly accurate predictor of model failure. 
1.  The high $D_{over}$ score correctly predicted the failure of Logistic Regression and SVM.
2.  The high $D_{noise}$ score correctly identified the need for Ensemble methods like XGBoost.
3.  The moderate $D_{imb}$ was secondary to the topological difficulty; even though we used a stratified split, the linear models could not "see" the minority class through the overlap.
   - To improve these scores further, the DDI analysis suggests that Feature Engineering or SMOTE-Tomek (to reduce overlap) would be more effective than simple hyperparameter tuning, as the primary bottleneck is the "smearing" of the two class distributions.
